In [1]:
import pandas as pd 
import ast
import numpy as np

In [2]:
%cd Thesis-FOS-BinaryClass-WSD/Ensemble Model/

c:\Users\Miguel\Documents\Project Source Files\IT Work\School\Thesis\Thesis-FOS-BinaryClass-WSD\Ensemble Model


c:\Users\Miguel\Documents\Project Source Files\IT Work\School\Thesis\.venv\Lib\site-packages\IPython\core\magics\osm.py:417: UserWarning: This is now an optional IPython functionality, setting dhist requires you to install the `pickleshare` library.
  self.shell.db['dhist'] = compress_dhist(dhist)[-100:]


# Load Models

## Sentence Transformer -- Sentence Embeddings


In [3]:
from sentence_transformers import SentenceTransformer
embedding_model = SentenceTransformer('sentence-transformers/LaBSE')

c:\Users\Miguel\Documents\Project Source Files\IT Work\School\Thesis\.venv\Lib\site-packages\sentence_transformers\cross_encoder\CrossEncoder.py:13: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from tqdm.autonotebook import tqdm, trange


# Processing

## Load Dataset

In [4]:
df_test = pd.read_excel('../../Dataset/Test_Set.xlsx')
df_train= pd.read_excel('../../Dataset/Train_Set.xlsx')
display(df_test.head(1),df_test.shape,df_train.head(1),df_train.shape)

,FOS,Word Sense,Verb,Non-Literal Usage,Literal Usage
0,anak sa sala,usa ka anak nga dili-konsejero,['sala'],Ang anak sa sala nga si Maria kay ginaatiman g...,Ang anak sa sala nagdula duol sa altar samtang...


(161, 5)

,FOS,Word Sense,Verb,Non-Literal Usage,Literal Usage
0,magpalapad og papel,aron magbaton ug dungog sa buhat sa uban,['magpalapad'],Ang iyang kauban sa trabaho pirmi magpalapad o...,Nagpalit siya og balayronon sa papel sa tindah...


(644, 5)

## Sentence Transformer

Removing the label column from the dataset for clarity

In [5]:
# df_train.drop(columns=['Is FOS'], inplace=True)
df = df_train.copy()

In [6]:
# df['Word Sense'] = df['Word Sense']
df.head(3)

,FOS,Word Sense,Verb,Non-Literal Usage,Literal Usage
0,magpalapad og papel,aron magbaton ug dungog sa buhat sa uban,['magpalapad'],Ang iyang kauban sa trabaho pirmi magpalapad o...,Nagpalit siya og balayronon sa papel sa tindah...
1,murag namiya og tai,Usa ka tawo nga mibiya sa dili-iasa nga walay ...,[],"""Ang silingan murag namiya og tai, kalit lang ...",Ang bata nagdali sa pagdagan kay murag namiya ...
2,makawat,mahimong ma-stolen,['makawat'],"Si Pedro makawat kaayo, dali ra siya madala bi...",Ang bata nakit-an nga makawat og mga mansanas ...


In [7]:
def GetEmbeddings(words:str) -> np.ndarray:
    return embedding_model.encode(words)

def ComputeSimilarity(embedding1,embedding2):
    return embedding_model.similarity(embedding1, embedding2)

### Generate Embeddings

> Generating the embeddings from the verb could be done another way than simply entering the list into the embeddings model

In [8]:
df['Sentence Embeddings'] = df['Word Sense'].apply(GetEmbeddings)
df['Verb Embeddings'] = df['Verb'].apply(GetEmbeddings)
df['Literal Usage Embeddings'] = df['Literal Usage'].apply(GetEmbeddings)
df['Non-Literal Usage Embeddings'] = df['Non-Literal Usage'].apply(GetEmbeddings)
df.head(3)

,FOS,Word Sense,Verb,Non-Literal Usage,Literal Usage,Sentence Embeddings,Verb Embeddings,Literal Usage Embeddings,Non-Literal Usage Embeddings
0,magpalapad og papel,aron magbaton ug dungog sa buhat sa uban,['magpalapad'],Ang iyang kauban sa trabaho pirmi magpalapad o...,Nagpalit siya og balayronon sa papel sa tindah...,"[0.0049795015, -0.034556165, 0.04411845, -0.05...","[-0.028341336, -0.006864522, -0.008147446, -0....","[-0.053077772, -0.042660423, -0.037503656, -0....","[-0.027987016, -0.007430709, 0.023322027, 0.01..."
1,murag namiya og tai,Usa ka tawo nga mibiya sa dili-iasa nga walay ...,[],"""Ang silingan murag namiya og tai, kalit lang ...",Ang bata nagdali sa pagdagan kay murag namiya ...,"[-0.03320765, -0.006950153, 0.012191928, 0.016...","[-0.02244177, 0.010921938, -0.016191624, -0.04...","[-0.012085079, 0.020437635, 0.029615637, -0.06...","[-0.01381951, 0.021733848, -0.01817845, 0.0346..."
2,makawat,mahimong ma-stolen,['makawat'],"Si Pedro makawat kaayo, dali ra siya madala bi...",Ang bata nakit-an nga makawat og mga mansanas ...,"[0.0023797662, -0.02209997, 0.027869346, -0.06...","[-0.0145964045, 0.008892309, -0.011957685, -0....","[-0.049912516, 0.06605863, 0.053893823, -0.060...","[-0.046755843, -0.0243083, -0.032696635, 0.030..."


### Compute Similarity Scores

In [9]:
df[:2].apply(lambda row: print(ComputeSimilarity(row['Sentence Embeddings'], row['Literal Usage Embeddings'])),axis=1)

tensor([[0.3356]])
tensor([[0.2567]])


0    None
1    None
dtype: object

In [10]:
df['Literal Similarity'] = df.apply(lambda row: ComputeSimilarity(row['Sentence Embeddings'], row['Literal Usage Embeddings']),axis=1)
df['Non-literal Similarity'] = df.apply(lambda row: ComputeSimilarity(row['Sentence Embeddings'], row['Non-Literal Usage Embeddings']),axis=1)

In [11]:
df.head(3)

,FOS,Word Sense,Verb,Non-Literal Usage,Literal Usage,Sentence Embeddings,Verb Embeddings,Literal Usage Embeddings,Non-Literal Usage Embeddings,Literal Similarity,Non-literal Similarity
0,magpalapad og papel,aron magbaton ug dungog sa buhat sa uban,['magpalapad'],Ang iyang kauban sa trabaho pirmi magpalapad o...,Nagpalit siya og balayronon sa papel sa tindah...,"[0.0049795015, -0.034556165, 0.04411845, -0.05...","[-0.028341336, -0.006864522, -0.008147446, -0....","[-0.053077772, -0.042660423, -0.037503656, -0....","[-0.027987016, -0.007430709, 0.023322027, 0.01...",[[tensor(0.3356)]],[[tensor(0.4529)]]
1,murag namiya og tai,Usa ka tawo nga mibiya sa dili-iasa nga walay ...,[],"""Ang silingan murag namiya og tai, kalit lang ...",Ang bata nagdali sa pagdagan kay murag namiya ...,"[-0.03320765, -0.006950153, 0.012191928, 0.016...","[-0.02244177, 0.010921938, -0.016191624, -0.04...","[-0.012085079, 0.020437635, 0.029615637, -0.06...","[-0.01381951, 0.021733848, -0.01817845, 0.0346...",[[tensor(0.2567)]],[[tensor(0.3334)]]
2,makawat,mahimong ma-stolen,['makawat'],"Si Pedro makawat kaayo, dali ra siya madala bi...",Ang bata nakit-an nga makawat og mga mansanas ...,"[0.0023797662, -0.02209997, 0.027869346, -0.06...","[-0.0145964045, 0.008892309, -0.011957685, -0....","[-0.049912516, 0.06605863, 0.053893823, -0.060...","[-0.046755843, -0.0243083, -0.032696635, 0.030...",[[tensor(0.2428)]],[[tensor(0.2405)]]


## PCA-Guided K-means


## Classification Models

# Output